# dARK local-ha: acceptance of a deposit lifecycle

This notebook accepts the maintained `local-ha` operator scenario, not a generic collection of local ports. It verifies the compact inventory and resolved topology, then performs an authority → ARK → publication → resolver lifecycle through the gateway actually exposed by the deployment.

Before running it, install and verify the scenario from the repository root:

```bash
venv/bin/python deploy.py install --inventory examples/operator-inventory/local-ha.json --verbose
venv/bin/python deploy.py verify --inventory examples/operator-inventory/local-ha.json
```

`local-ha` creates four QBFT validators and two Kubo/IPFS Cluster peers. They all run on the local machine, so two storage copies are verified but host-failure tolerance is deliberately **not** claimed. The only HTTP listener used from the host is the loopback gateway at `http://localhost`. Admin and Store remain private; the notebook reaches them from their own containers without publishing extra ports.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
import time
import uuid
from pathlib import Path
from pprint import pprint

import requests

def find_repository_root() -> Path:
    configured = os.getenv('DARK_DEPLOYER_ROOT')
    candidates = ([Path(configured).expanduser()] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path.home() / 'source' / 'dark-deployer']
    for candidate in candidates:
        candidate = candidate.resolve()
        if (candidate / 'deploy.py').is_file() and (candidate / 'deployment_v3').is_dir():
            return candidate
    raise RuntimeError('Cannot find dark-deployer. Set DARK_DEPLOYER_ROOT to its absolute path.')

ROOT = find_repository_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


INVENTORY = ROOT / os.getenv('DARK_INVENTORY', 'examples/operator-inventory/local-ha.json')
GATEWAY_BASE_URL = os.getenv('DARK_GATEWAY_BASE_URL', 'http://localhost').rstrip('/')
AUTHORITY_ID = os.getenv('AUTHORITY_ID', f'local-ha-notebook-{uuid.uuid4().hex[:12]}')
NAAN = os.getenv('NAAN', '12345').strip()
PLATFORM_ITEM_ID = os.getenv('PLATFORM_ITEM_ID', f'local-ha-item-{uuid.uuid4().hex[:12]}')
TARGET_URL = os.getenv('TARGET_URL', f'https://repository.example.org/items/{PLATFORM_ITEM_ID}')
POLL_INTERVAL_SECONDS = float(os.getenv('POLL_INTERVAL_SECONDS', '2'))
POLL_TIMEOUT_SECONDS = int(os.getenv('POLL_TIMEOUT_SECONDS', '180'))

print('inventory:', INVENTORY)
print('gateway:', GATEWAY_BASE_URL)
print('authority:', AUTHORITY_ID)
print('item:', PLATFORM_ITEM_ID)

## 1. Resolve and inspect the declared scenario

Resolution is pure: it contacts neither Docker nor remote hosts. The assertions below are intentional guardrails against accidentally running the notebook against a different topology.

In [ ]:
from deployment_v3.inventory_resolver import resolve_inventory_path
from deployment_v3.planner import build_plan
from deployment_v3.availability import analyze
from deployment_v3.network import internal_port

resolution = resolve_inventory_path(INVENTORY)
resolved = resolution.document
plan = build_plan(INVENTORY)
availability = analyze(plan)

assert resolved['deployment']['id'] == 'dark-operator-local-ha'
assert availability.validators == 4
assert availability.quorum == 3
assert availability.validator_failures_tolerated == 1
assert availability.storage_peers == 2
assert availability.storage_target_replicas == 2
assert availability.storage_publish_after_replicas == 1
assert availability.primary_rpc == 'rpc01'
assert len(availability.validator_machines) == 1
assert 'Storage replication shares a machine failure domain.' in availability.warnings

print('validators:', availability.validators, 'quorum:', availability.quorum)
print('storage peers:', availability.storage_peers, 'replication:', f'{availability.storage_publish_after_replicas}/{availability.storage_target_replicas}')
print('primary RPC:', availability.primary_rpc)
print('availability warnings:')
for warning in availability.warnings:
    print(' -', warning)

In [ ]:
services = resolved['services']
gateway_id, gateway = next((service_id, service) for service_id, service in services.items() if service['type'] == 'edge-proxy')
gateway_exposure = gateway['exposure']

assert gateway_exposure['mode'] == 'loopback'
assert gateway_exposure['port'] == 80
assert not services['admin-api'].get('exposure')
assert not services['store-api'].get('exposure')

MINTER_API_V1 = f'{GATEWAY_BASE_URL}/api/v1'
RESOLVER_ARK_BASE = f'{GATEWAY_BASE_URL}/'
MINTER_HEADERS = {'X-Authority-Id': AUTHORITY_ID}

print(f'public gateway service: {gateway_id} at 127.0.0.1:80')
print('public Minter API:', MINTER_API_V1)
print('public Resolver ARK base:', RESOLVER_ARK_BASE)
print('Admin and Store are private Docker-only services.')

## 2. Check the running deployment

The public Minter worker status proves that the gateway path is usable. For the two private services used by the acceptance flow, a short Python request runs inside the relevant already-running container. This does not publish ports, create networks, or alter the rendered deployment.

In [ ]:
def container_id(service_id: str) -> str:
    service = plan.service(service_id)
    group_id = next(group.id for group in plan.groups if service_id in group.service_ids)
    project = f'{plan.deployment_id}-{service.machine_id}-{group_id}'
    result = subprocess.run(
        ['docker', 'ps', '--filter', f'label=com.docker.compose.project={project}',
         '--filter', f'label=com.docker.compose.service={service_id}', '--format', '{{.ID}}'],
        cwd=ROOT, text=True, capture_output=True, check=True,
    )
    matches = [line for line in result.stdout.splitlines() if line]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one running {service_id} container in {project}; found {matches or 'none'}. Run install and verify first.")
    return matches[0]

def private_request(service_id: str, method: str, path: str, *, payload: dict | None = None) -> tuple[int, dict | str]:
    service = plan.service(service_id)
    port = internal_port(service)
    request_spec = {'method': method, 'url': f'http://127.0.0.1:{port}{path}', 'payload': payload}
    script = '''\
import json, sys, urllib.error, urllib.request
spec = json.load(sys.stdin)
body = None if spec['payload'] is None else json.dumps(spec['payload']).encode()
request = urllib.request.Request(spec['url'], data=body, method=spec['method'])
if body is not None:
    request.add_header('Content-Type', 'application/json')
try:
    response = urllib.request.urlopen(request, timeout=30)
    status, text = response.status, response.read().decode()
except urllib.error.HTTPError as error:
    status, text = error.code, error.read().decode()
print(json.dumps({'status': status, 'body': text}))
'''
    result = subprocess.run(
        ['docker', 'exec', '-i', container_id(service_id), 'python', '-c', script],
        input=json.dumps(request_spec), cwd=ROOT, text=True, capture_output=True, check=True,
    )
    response = json.loads(result.stdout)
    try:
        body = json.loads(response['body'])
    except json.JSONDecodeError:
        body = response['body']
    return response['status'], body

worker_status = requests.get(f'{MINTER_API_V1}/worker/status', timeout=30)
print('gateway worker status:', worker_status.status_code)
pprint(worker_status.json())
worker_status.raise_for_status()
for worker_name in ('metadata', 'replication', 'chain'):
    assert worker_status.json()['workers'][worker_name]['alive'] is True

admin_health, _ = private_request('admin-api', 'GET', '/health')
store_health, _ = private_request('store-api', 'GET', '/health')
assert admin_health == 200
assert store_health == 200
print('private Admin and Store health checks passed')

## 3. Provision an isolated test authority

A unique authority is created inside this disposable local deployment and authorized for the selected NAAN. This is the only setup mutation made by the notebook.

In [ ]:
status, body = private_request('admin-api', 'POST', '/api/v1/admin/authority', payload={
    'uuid': AUTHORITY_ID, 'naans': [], 'fund_amount_eth': 0.05,
})
assert status in {200, 201, 409}, body

status, body = private_request('admin-api', 'POST', f'/api/v1/admin/authority/{AUTHORITY_ID}/authorize-naan', payload={'naan': NAAN})
assert status == 200, body

authority_naans = requests.get(f'{MINTER_API_V1}/authority/{AUTHORITY_ID}/naans', timeout=30)
authority_naans.raise_for_status()
assert NAAN in authority_naans.json()['naans']
print('authority authorized:', authority_naans.json())

## 4. Reserve, publish, and resolve an ARK through the gateway

In [ ]:
def get_ark(ark: str) -> dict:
    response = requests.get(f'{MINTER_API_V1}/arks/{ark}', headers=MINTER_HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()

def wait_until_published(ark: str) -> dict:
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    last = None
    while time.time() < deadline:
        last = get_ark(ark)
        print(f"{time.strftime('%H:%M:%S')} state={last['state']} level1={last.get('level1_cid')}")
        if last['state'] == 'P':
            return last
        time.sleep(POLL_INTERVAL_SECONDS)
    raise TimeoutError(f'ARK {ark} did not reach PUBLISHED. Last response: {last}')

reserve_payload = {'authority_id': AUTHORITY_ID, 'naan': NAAN, 'items': [{'client_item_id': PLATFORM_ITEM_ID}]}
reserve = requests.post(f'{MINTER_API_V1}/arks/batch', headers=MINTER_HEADERS, json=reserve_payload, timeout=60)
reserve.raise_for_status()
reservation = reserve.json()['results'][0]
assert reservation['state'] == 'R', reservation
ARK = reservation['ark']

retry = requests.post(f'{MINTER_API_V1}/arks/batch', headers=MINTER_HEADERS, json=reserve_payload, timeout=60)
retry.raise_for_status()
assert retry.json()['results'][0]['ark'] == ARK
print('reserved ARK:', ARK)

In [ ]:
title = 'Objeto de aceptación local-ha'
level1 = {
    'title': title, 'authors': ['Equipo dARK'], 'year': 2026,
    'publisher': 'Repositorio de demostración', 'resource_type': 'article', 'language': 'es',
    'abstract': 'Registro creado por la aceptación del escenario local-ha.',
    'subjects': ['interoperabilidad', 'aceptación'],
    'alternate_identifiers': [{'schema': 'platform-item-id', 'value': PLATFORM_ITEM_ID}],
    'alternate_urls': [TARGET_URL],
}
level2 = f'<oai_dc:dc xmlns:oai_dc="http://www.openarchives.org/OAI/2.0/oai_dc/" xmlns:dc="http://purl.org/dc/elements/1.1/"><dc:identifier>{TARGET_URL}</dc:identifier><dc:title>{title}</dc:title></oai_dc:dc>'
stage = requests.put(
    f'{MINTER_API_V1}/arks/{ARK}', headers=MINTER_HEADERS,
    json={'authority_id': AUTHORITY_ID, 'target': TARGET_URL, 'minimal_metadata': level1,
          'original_metadata': level2, 'metadata_schema': 'oai_dc', 'metadata_media_type': 'application/xml'},
    timeout=60,
)
stage.raise_for_status()
assert stage.json()['state'] == 'D', stage.json()
published = wait_until_published(ARK)
assert published['target'] == TARGET_URL
assert published['level1_cid'] and published['level2_cid']
pprint(published)

## 5. Prove object replication and public resolution

Store's per-CID status remains private by design, so it is queried from its container. The final request enters only through the public resolver gateway path and must redirect to the original target.

In [ ]:
def wait_for_replication(cid: str) -> dict:
    target = availability.storage_target_replicas
    deadline = time.time() + POLL_TIMEOUT_SECONDS
    last = None
    while time.time() < deadline:
        status, last = private_request('store-api', 'GET', f'/v1/status/{cid}')
        assert status == 200, last
        replication = last.get('replication', {})
        print(f"replication {cid}: total={replication.get('total_replicas')} assigned={replication.get('assigned_replicas')} pinning={replication.get('pinning_replicas')} queued={replication.get('queued_replicas')} errors={replication.get('error_replicas')}")
        if replication.get('error_replicas', 0) > 0:
            raise RuntimeError(f'CID {cid} replication failed: {last}')
        if replication.get('total_replicas', 0) >= target:
            return last
        time.sleep(POLL_INTERVAL_SECONDS)
    raise TimeoutError(f'CID {cid} did not reach {target} replicas. Last status: {last}')

for label, cid in (('level1', published['level1_cid']), ('level2', published['level2_cid'])):
    replication = wait_for_replication(cid)
    print(label, cid)
    pprint(replication)

resolved_redirect = requests.get(f'{RESOLVER_ARK_BASE}{ARK}', allow_redirects=False, timeout=30)
assert resolved_redirect.status_code in {302, 307}, (resolved_redirect.status_code, resolved_redirect.text)
assert resolved_redirect.headers['location'] == TARGET_URL

print(f'Acceptance passed: {PLATFORM_ITEM_ID} -> {ARK} -> {TARGET_URL}')
print('Topology: 4 validators (quorum 3), 1 primary RPC, 2 storage copies on one local host.')